# 📊 05 — Evaluasi Final & Perbandingan Semua Metode
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

Notebook ini merupakan **tahap akhir analisis** yang mengkonsolidasikan seluruh hasil evaluasi
dari semua metode, menghasilkan tabel dan visualisasi siap pakai untuk laporan skripsi.

### Metode yang Dibandingkan

| # | Metode | Notebook |
|---|--------|----------|
| 1 | TF-IDF + Cosine Similarity | `03_tfidf_baseline` |
| 2 | Indo Sentence-BERT Embedding | `04A_bert_embedding` |
| 3 | IndoBERT Mean Pooling | `04A_bert_embedding` |
| 4 | Indo SBERT + SetFit Fine-tuning *(Opsi B)* | `04B_setfit` |

> ⚠️ Pastikan semua notebook sebelumnya sudah dijalankan dan `results/evaluation.csv` sudah terisi.

---
## 🔧 LANGKAH 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))

import config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family']  = 'DejaVu Sans'
plt.rcParams['figure.dpi']   = 120
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print('✅ Setup selesai.')

---
## 📌 LANGKAH 1 — Load Semua Hasil Evaluasi

In [ ]:
df_eval = pd.read_csv(config.FILE_EVALUATION)
df_skripsi = pd.read_csv(config.FILE_SKRIPSI_CLEAN)
df_dosen   = pd.read_csv(config.FILE_DOSEN_CLEAN)

print('✅ Data evaluasi dimuat.')
print(f'   Jumlah metode yang dievaluasi: {len(df_eval)}')
print()
print('📋 Tabel Evaluasi Lengkap:')
print('='*65)
print(df_eval[['metode','top1_accuracy','top3_accuracy','top5_accuracy','n_test']].to_string(index=False))
print('='*65)

---
## 📌 LANGKAH 2 — Tabel Perbandingan Formal (Siap Skripsi)

In [ ]:
# ─── BUAT TABEL FORMAL ───────────────────────────────────────────
# Format seperti di jurnal/paper

df_tabel = df_eval[['metode','top1_accuracy','top3_accuracy','top5_accuracy']].copy()
df_tabel.columns = ['Metode', 'Top-1 Accuracy (%)', 'Top-3 Accuracy (%)', 'Top-5 Accuracy (%)']
df_tabel = df_tabel.reset_index(drop=True)
df_tabel.index = df_tabel.index + 1  # mulai dari 1

# Tandai nilai tertinggi per kolom
for col in ['Top-1 Accuracy (%)','Top-3 Accuracy (%)','Top-5 Accuracy (%)']:
    max_val = df_tabel[col].max()
    df_tabel[col] = df_tabel[col].apply(
        lambda x: f'**{x:.2f}**' if x == max_val else f'{x:.2f}'
    )

print('📋 TABEL PERBANDINGAN METODE (format skripsi):')
print('   (nilai tertinggi ditandai **bold**)')
print()
print(df_tabel.to_markdown())

In [ ]:
# ─── SIMPAN TABEL KE CSV (siap copy ke Word/LaTeX) ───────────────
tabel_path = os.path.join(config.RESULTS_DIR, 'tabel_perbandingan_final.csv')
df_eval.to_csv(tabel_path, index=False, encoding='utf-8-sig')
print(f'💾 Tabel perbandingan tersimpan: {tabel_path}')

---
## 📌 LANGKAH 3 — Visualisasi Komprehensif

In [ ]:
# ─── GRAFIK 1: GROUPED BAR — TOP-K ACCURACY SEMUA METODE ─────────
df_plot  = pd.read_csv(config.FILE_EVALUATION)
n_metode = len(df_plot)
ks       = ['Top-1', 'Top-3', 'Top-5']
cols     = ['top1_accuracy','top3_accuracy','top5_accuracy']
palette  = ['#1565C0','#2E7D32','#E65100','#6A1B9A']
x        = np.arange(len(ks))
w        = 0.8 / n_metode

fig, ax = plt.subplots(figsize=(13, 6))
for i, (_, row) in enumerate(df_plot.iterrows()):
    vals   = [row[c] for c in cols]
    offset = (i - n_metode/2 + 0.5) * w
    bars   = ax.bar(x + offset, vals, width=w*0.9,
                    color=palette[i % len(palette)], label=row['metode'],
                    edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(ks, fontsize=13)
ax.set_ylim(0, 118)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_xlabel('Nilai K', fontsize=12)
ax.set_title('Perbandingan Top-K Accuracy Semua Metode\nSistem Rekomendasi Dosen Pembimbing',
             fontweight='bold', fontsize=13, pad=15)
ax.legend(loc='upper left', fontsize=8.5, framealpha=0.9, ncol=1)
ax.axhline(100, color='gray', linestyle='--', alpha=0.3, label='Batas Maksimum')
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'fig1_topk_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()
print('📊 Grafik 1 tersimpan: results/fig1_topk_comparison.png')

In [ ]:
# ─── GRAFIK 2: RADAR CHART — PROFIL PERFORMA TIAP METODE ─────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.gridspec as gridspec

categories = ['Top-1', 'Top-3', 'Top-5']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})

for i, (_, row) in enumerate(df_plot.iterrows()):
    vals = [row[c] for c in cols]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, linestyle='solid',
            label=row['metode'], color=palette[i % len(palette)])
    ax.fill(angles, vals, alpha=0.08, color=palette[i % len(palette)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20%','40%','60%','80%','100%'], fontsize=8, color='gray')
ax.set_title('Radar Chart — Profil Akurasi Per Metode',
             fontweight='bold', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
ax.set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'fig2_radar_chart.png'), dpi=200, bbox_inches='tight')
plt.show()
print('📊 Grafik 2 tersimpan: results/fig2_radar_chart.png')

In [ ]:
# ─── GRAFIK 3: HEATMAP — AKURASI METODE × K ──────────────────────
heatmap_data = df_plot.set_index('metode')[cols]
heatmap_data.columns = ['Top-1','Top-3','Top-5']

fig, ax = plt.subplots(figsize=(8, max(3, len(df_plot)*0.9)))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlGn',
            vmin=0, vmax=100, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Accuracy (%)', 'shrink': 0.8},
            ax=ax, annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Heatmap Top-K Accuracy Semua Metode', fontweight='bold', fontsize=13, pad=12)
ax.set_xlabel('Nilai K', fontsize=11)
ax.set_ylabel('')
ax.tick_params(axis='y', rotation=0, labelsize=9)
ax.tick_params(axis='x', labelsize=11)

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'fig3_heatmap_accuracy.png'), dpi=200, bbox_inches='tight')
plt.show()
print('📊 Grafik 3 tersimpan: results/fig3_heatmap_accuracy.png')

---
## 📌 LANGKAH 4 — Analisis Mendalam per Dosen

In [ ]:
# ─── LOAD MODEL TF-IDF UNTUK ANALISIS PER DOSEN ──────────────────
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

with open(config.FILE_TFIDF_MODEL, 'rb') as f:
    tfidf_obj = pickle.load(f)

vectorizer   = tfidf_obj['vectorizer']
tfidf_dosen  = tfidf_obj['tfidf_dosen']
NAMA_DOSEN   = tfidf_obj['nama_dosen']

# Siapkan data test yang sama
dosen_valid = set(NAMA_DOSEN)
df_eval_sk  = df_skripsi[df_skripsi['pembimbing'].isin(dosen_valid)].copy().reset_index(drop=True)
_, df_test  = train_test_split(df_eval_sk, test_size=0.2, random_state=42, stratify=df_eval_sk['pembimbing'])

print(f'✅ Model TF-IDF dimuat. Data test: {len(df_test)} skripsi')

In [ ]:
# ─── ANALISIS TOP-1 ACCURACY PER DOSEN (TF-IDF) ──────────────────
per_dosen = {nama: {'total': 0, 'benar': 0} for nama in NAMA_DOSEN}

for _, row in df_test.iterrows():
    gt       = row['pembimbing']
    query    = vectorizer.transform([row['teks_bersih']])
    scores   = cosine_similarity(query, tfidf_dosen).flatten()
    pred     = NAMA_DOSEN[np.argmax(scores)]

    per_dosen[gt]['total'] += 1
    if pred == gt:
        per_dosen[gt]['benar'] += 1

# Hitung akurasi per dosen
df_per_dosen = pd.DataFrame([
    {'dosen': k, 'total': v['total'], 'benar': v['benar'],
     'accuracy': round(v['benar']/v['total']*100, 1) if v['total'] > 0 else 0}
    for k, v in per_dosen.items() if v['total'] > 0
]).sort_values('accuracy', ascending=True).reset_index(drop=True)

print('📋 Top-1 Accuracy per Dosen (TF-IDF):')
print('='*65)
for _, row in df_per_dosen.iterrows():
    bar = '█' * int(row['accuracy'] / 5)
    print(f'  {row["dosen"][:38]:<40} {row["accuracy"]:5.1f}%  {bar}')
print('='*65)

In [ ]:
# ─── GRAFIK 4: AKURASI PER DOSEN ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, max(5, len(df_per_dosen)*0.5)))

colors_bar = ['#EF5350' if a < 50 else '#FF9800' if a < 75 else '#66BB6A'
               for a in df_per_dosen['accuracy']]
bars = ax.barh(df_per_dosen['dosen'].apply(lambda x: x.split(',')[0]),
               df_per_dosen['accuracy'], color=colors_bar, edgecolor='white')

for bar, row in zip(bars, df_per_dosen.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{row.accuracy:.0f}% ({row.benar}/{row.total})',
            va='center', fontsize=9)

ax.set_xlim(0, 120)
ax.axvline(50,  color='#FF9800', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(75,  color='#66BB6A', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(100, color='gray',    linestyle='--', alpha=0.3, linewidth=1)
ax.set_xlabel('Top-1 Accuracy (%)', fontsize=11)
ax.set_title('Top-1 Accuracy per Dosen — TF-IDF + Cosine Similarity',
             fontweight='bold', fontsize=12, pad=12)

legend_patches = [
    mpatches.Patch(color='#EF5350', label='< 50% (Rendah)'),
    mpatches.Patch(color='#FF9800', label='50–74% (Sedang)'),
    mpatches.Patch(color='#66BB6A', label='≥ 75% (Baik)'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'fig4_accuracy_per_dosen.png'), dpi=200, bbox_inches='tight')
plt.show()
print('📊 Grafik 4 tersimpan.')

---
## 📌 LANGKAH 5 — Analisis Kesalahan (Error Analysis)

In [ ]:
# ─── CONFUSION MATRIX DOSEN (TF-IDF Top-1) ───────────────────────
gt_list   = []
pred_list = []

for _, row in df_test.iterrows():
    gt    = row['pembimbing']
    query = vectorizer.transform([row['teks_bersih']])
    scores = cosine_similarity(query, tfidf_dosen).flatten()
    pred  = NAMA_DOSEN[np.argmax(scores)]
    gt_list.append(gt)
    pred_list.append(pred)

# Buat confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

labels_uniq = sorted(dosen_valid)
cm = confusion_matrix(gt_list, pred_list, labels=labels_uniq)

nama_pendek = [n.split(',')[0].split('.')[-1].strip()[:14] for n in labels_uniq]

fig, ax = plt.subplots(figsize=(max(10, len(labels_uniq)), max(8, len(labels_uniq)*0.7)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nama_pendek, yticklabels=nama_pendek,
            linewidths=0.4, linecolor='white', ax=ax,
            cbar_kws={'label': 'Jumlah Prediksi'})
ax.set_xlabel('Prediksi (Dosen yang Direkomendasikan)', fontsize=11)
ax.set_ylabel('Aktual (Dosen Pembimbing Asli)', fontsize=11)
ax.set_title('Confusion Matrix — TF-IDF Top-1 Prediction',
             fontweight='bold', fontsize=12, pad=12)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'fig5_confusion_matrix.png'), dpi=200, bbox_inches='tight')
plt.show()
print('📊 Grafik 5 (Confusion Matrix) tersimpan.')

In [ ]:
# ─── CONTOH KASUS SALAH PREDIKSI (Top-1) ─────────────────────────
errors = [
    {'judul': df_test.iloc[i]['judul'], 'gt': gt_list[i], 'pred': pred_list[i]}
    for i in range(len(gt_list)) if gt_list[i] != pred_list[i]
]

print(f'🔎 Jumlah kesalahan Top-1: {len(errors)} dari {len(df_test)} skripsi')
print(f'   Accuracy Top-1        : {(1 - len(errors)/len(df_test))*100:.2f}%')
print()

if errors:
    print('📋 10 Contoh Kasus Salah Prediksi:')
    print('-' * 100)
    for i, err in enumerate(errors[:10], 1):
        print(f'  [{i:2}] Judul : {err["judul"][:70]}')
        print(f'       GT    : {err["gt"]}')
        print(f'       Pred  : {err["pred"]}')
        print()

---
## 📌 LANGKAH 6 — Ringkasan & Temuan Kunci

In [ ]:
# ─── CETAK RINGKASAN EKSEKUTIF ────────────────────────────────────
df_res = pd.read_csv(config.FILE_EVALUATION)

best_top1 = df_res.loc[df_res['top1_accuracy'].idxmax()]
best_top3 = df_res.loc[df_res['top3_accuracy'].idxmax()]
best_top5 = df_res.loc[df_res['top5_accuracy'].idxmax()]

print('=' * 68)
print('                    📊 RINGKASAN TEMUAN KUNCI')
print('=' * 68)
print()
print(f'  Dataset  : {len(df_skripsi)} skripsi | {len(df_dosen)} dosen')
print(f'  Test set : {df_res["n_test"].iloc[0]} skripsi (20% dari data valid)')
print()
print('  PERFORMA TERBAIK PER METRIK:')
print(f'  ┌─────────┬───────────────────────────────────────────────────┐')
print(f'  │ Metrik  │ Metode Terbaik                          Accuracy │')
print(f'  ├─────────┼───────────────────────────────────────────────────┤')
print(f'  │ Top-1   │ {best_top1["metode"][:42]:<42} {best_top1["top1_accuracy"]:6.2f}% │')
print(f'  │ Top-3   │ {best_top3["metode"][:42]:<42} {best_top3["top3_accuracy"]:6.2f}% │')
print(f'  │ Top-5   │ {best_top5["metode"][:42]:<42} {best_top5["top5_accuracy"]:6.2f}% │')
print(f'  └─────────┴───────────────────────────────────────────────────┘')
print()

# Perbandingan BERT vs TF-IDF
tfidf_row = df_res[df_res['metode'].str.contains('TF-IDF')].iloc[0]
best_bert = df_res[~df_res['metode'].str.contains('TF-IDF')].nlargest(1,'top1_accuracy').iloc[0]
delta = best_bert['top1_accuracy'] - tfidf_row['top1_accuracy']

print(f'  PERBANDINGAN TF-IDF vs BERT Terbaik (Top-1):')
if delta > 0:
    print(f'  → BERT unggul {delta:.2f}% dibanding TF-IDF')
elif delta < 0:
    print(f'  → TF-IDF masih unggul {abs(delta):.2f}% dibanding BERT')
else:
    print(f'  → Performa TF-IDF dan BERT terbaik setara')
print()
print('=' * 68)

In [ ]:
# ─── SIMPAN RINGKASAN KE FILE TEKS ───────────────────────────────
summary_path = os.path.join(config.RESULTS_DIR, 'ringkasan_evaluasi.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('RINGKASAN HASIL EVALUASI\n')
    f.write('Sistem Rekomendasi Dosen Pembimbing Berbasis NLP\n')
    f.write('Teknik Informatika — Universitas Lampung\n')
    f.write('='*60 + '\n\n')
    for _, row in df_res.iterrows():
        f.write(f'Metode  : {row["metode"]}\n')
        f.write(f'Top-1   : {row["top1_accuracy"]:.2f}%\n')
        f.write(f'Top-3   : {row["top3_accuracy"]:.2f}%\n')
        f.write(f'Top-5   : {row["top5_accuracy"]:.2f}%\n')
        f.write('-'*40 + '\n')

print(f'💾 Ringkasan evaluasi tersimpan: {summary_path}')
print()
print('📁 Semua output hasil evaluasi:')
for f in sorted(os.listdir(config.RESULTS_DIR)):
    fpath = os.path.join(config.RESULTS_DIR, f)
    size  = os.path.getsize(fpath) / 1024
    print(f'   {f:<45} ({size:.1f} KB)')

---
## ✅ Selesai — Ringkasan Notebook 05

| Output | Keterangan |
|--------|------------|
| `fig1_topk_comparison.png` | Grouped bar chart semua metode |
| `fig2_radar_chart.png` | Radar chart profil performa |
| `fig3_heatmap_accuracy.png` | Heatmap akurasi |
| `fig4_accuracy_per_dosen.png` | Akurasi per dosen pembimbing |
| `fig5_confusion_matrix.png` | Confusion matrix TF-IDF |
| `tabel_perbandingan_final.csv` | Tabel siap skripsi |
| `ringkasan_evaluasi.txt` | Temuan kunci |

### 🗺️ Langkah Terakhir:
> **`06_prototype_app.ipynb`** — Prototype aplikasi web dengan Streamlit